In [ ]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../..")

from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aemri_articulatory_features

In [2]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_aemri.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])

In [3]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,2.28270,2.33270,2.33270,2.43270,h,ɝ,h,her,0.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
1,2.43270,2.64270,2.64270,2.67270,ɔ,b,ɔ,auburn,1.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
2,2.64270,2.67270,2.67270,2.82270,b,ɝ,ɔ_b,auburn,1.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
3,2.67270,2.82270,2.82270,2.86270,ɝ,n,ɔ_b_ɝ,auburn,1.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
4,2.86270,2.96270,2.96270,3.19270,h,ɛ,h,hair,2.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
...,...,...,...,...,...,...,...,...,...,...,...,...
98930,22.56169,22.59172,22.59172,22.63176,m,u,m,MOVIES,36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
98931,22.59172,22.63176,22.63176,22.68181,u,v,m_u,MOVIES,36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
98932,22.63176,22.68181,22.68181,22.88201,v,i,m_u_v,MOVIES,36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
98933,22.68181,22.88201,22.88201,23.12225,i,z,m_u_v_i,MOVIES,36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1


In [ ]:
tokens = {
    "start": [],
    "end": [],
    "syllable": [],
    "art_f": [],
    "ac_f": [],
    "subject": [],
}

metadata = set(
    [(start, end, target, art_f, ac_f, subject) for start, end, target, art_f, ac_f, subject in zip(df["syllable1_start"], df["syllable1_end"], df["syllable1"], df["art_f"], df["ac_f"], df["subject"])] + 
    [(start, end, target, art_f, ac_f, subject) for start, end, target, art_f, ac_f, subject in zip(df["syllable2_start"], df["syllable2_end"], df["syllable2"], df["art_f"], df["ac_f"], df["subject"])]
)

metadata = sorted(list(metadata), key=lambda x: (x[3], x[4], x[5]))

for start, end, target, art_f, ac_f, subject in metadata:
    tokens["start"].append(start)
    tokens["end"].append(end)
    tokens["syllable"].append(target)
    tokens["art_f"].append(art_f)
    tokens["ac_f"].append(ac_f)
    tokens["subject"].append(subject)

In [5]:
token_df = pd.DataFrame(tokens)

In [6]:
token_df

,start,end,phone,art_f,ac_f,subject
0,2.28270,2.33270,h,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
1,2.43270,2.64270,ɔ,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
2,2.64270,2.67270,b,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
3,2.67270,2.82270,ɝ,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
4,2.86270,2.96270,h,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
...,...,...,...,...,...,...
129379,21.77090,21.80093,m,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
129380,21.92105,21.95108,z,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
129381,22.53166,22.56169,m,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
129382,22.88201,23.12225,z,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1


In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from utils.models import VideoAutoencoder, SequenceAutoencoder

In [8]:
def latent_stats(z):
    # z: (B, latent_dim)
    return {
        "latent_var_mean": z.var(dim=0).mean().item(),
        "latent_var_min":  z.var(dim=0).min().item(),
        "latent_var_max":  z.var(dim=0).max().item(),
    }

def batch_diversity(x):
    # x: (B, ...)
    x_flat = x.reshape(x.size(0), -1)
    return x_flat.std(dim=0).mean().item()

def visualize(clean, recon, name="Reconstruction"):
    import matplotlib.pyplot as plt

    try:
        # Get both channels, frame 0 of first batch element
        clean_mri= clean[0, 0, 0].cpu().numpy()
    except RuntimeError:
        clean_mri = clean[0, 0, 0].detach().cpu().numpy()

    recon_mri = recon[0, 0, 0].detach().cpu().numpy()

    plt.figure(figsize=(10, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(clean_mri, cmap='gray')
    plt.title("Original (MRI)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(recon_mri, cmap='gray')
    plt.title(f"{name} (MRI)")
    plt.axis('off')

    plt.tight_layout()
    plt.savefig(f"video_visualization/aemri/{name.lower()}.png", dpi=300)
    plt.close("all")

def combined_loss(y_true, y_pred, warped_x, criterion, plot=False, ratio=1):
    diff_true = y_true[:, :, 1:, :, :] - y_true[:, :, :-1, :, :]
    diff_pred = y_pred[:, :, 1:, :, :] - y_pred[:, :, :-1, :, :]
    loss = criterion(diff_true, diff_pred) * (1 - ratio) + criterion(y_true, y_pred) * ratio

    if plot:
        visualize(diff_true, diff_pred, name="Reconstruction (differential)")
        visualize(y_true, y_pred, name="Reconstruction")
        visualize(y_true, warped_x, name="Transformation")

    return loss

In [9]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

BATCH_SIZE = 4

In [ ]:
log = {
    "batch_idx": [],
    "loss": [],
}

art_model = VideoAutoencoder(
    input_shape=(5, 68, 68),
    latent_dim=1024,
    use_mlp=True,
    grid_sample_cpu_fallback=True,
).to(device)

art_criterion = nn.MSELoss()
art_optimizer = torch.optim.Adam(art_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

art_model.train()

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    
    articulatory_features = get_aemri_articulatory_features(art_f, start, end)

    if articulatory_features is None:
        continue

    batch.append(
        torch.tensor(articulatory_features, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1

    art_batch = torch.stack(batch).to(device)  # (B,T,P,C)
    batch.clear()

    # ===== Articulatory =====
    art_out, art_warped_x, art_z = art_model(art_batch, return_warped_x=True, return_latent=True)
    art_loss = combined_loss(art_batch, art_out, art_warped_x, art_criterion, plot=batch_idx % 30 == 0, ratio=0.5)

    art_optimizer.zero_grad()
    art_loss.backward()
    art_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(art_loss.item())
    pbar.set_description(
        f"{batch_idx}: ART {art_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if (batch_idx) % 300 == 0:
        with torch.no_grad():
            art_stats = latent_stats(art_z)

        print(
            f"[ART] latent var mean={art_stats['latent_var_mean']:.4e}, "
            f"min={art_stats['latent_var_min']:.4e}"
        )

        print(
            f"[ART] input std={batch_diversity(art_batch):.4e}, "
            f"recon std={batch_diversity(art_out):.4e}"
        )

        torch.save(art_model.state_dict(), "ckpts/aemri/art_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/aemri/art_model_training_log.csv", index=False)

torch.save(art_model.state_dict(), "ckpts/aemri/art_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/aemri/art_model_training_log.csv", index=False)

300: ART 0.3035:   1%|          | 1196/129384 [00:54<1:26:14, 24.77it/s]

[ART] latent var mean=8.1664e+03, min=9.1396e-02
[ART] input std=4.1277e-01, recon std=4.7094e-02


600: ART 0.2895:   2%|▏         | 2400/129384 [01:48<3:28:43, 10.14it/s]

[ART] latent var mean=3.2170e+04, min=2.1764e-01
[ART] input std=3.8939e-01, recon std=6.3191e-02


900: ART 0.3051:   3%|▎         | 3600/129384 [02:49<3:46:36,  9.25it/s]

...

31800: ART 0.2794:  98%|█████████▊| 127200/129384 [1:34:45<03:14, 11.25it/s]

[ART] latent var mean=2.8608e+10, min=8.3487e+04
[ART] input std=4.0678e-01, recon std=8.9898e-02


32100: ART 0.2871:  99%|█████████▉| 128396/129384 [1:35:35<00:36, 27.03it/s]

[ART] latent var mean=1.7118e+10, min=4.3581e+05
[ART] input std=4.4282e-01, recon std=1.0340e-01


32346: ART 0.2858: 100%|██████████| 129384/129384 [1:36:20<00:00, 22.38it/s]


In [21]:
def temporal_diff(x):
    """
    x: (B, T, ...)
    returns: (B, T-1, ...)
    """
    return x[:, 1:] - x[:, :-1]

def combined_loss(x_hat, x, alpha=0.5):
    """
    alpha: weight on static loss (delta weight = 1-alpha)
    """
    # static loss
    loss_static = F.mse_loss(x_hat, x)

    # delta loss
    dx_hat = temporal_diff(x_hat)
    dx = temporal_diff(x)

    loss_delta = F.mse_loss(dx_hat, dx)

    return alpha * loss_static + (1 - alpha) * loss_delta, loss_static, loss_delta

In [22]:
BATCH_SIZE = 4

In [ ]:
log = {
    "batch_idx": [],
    "loss": [],
}
ac_model = SequenceAutoencoder(
    feature_shape=30,
    latent_dim=1024,
).to(device)

ac_criterion = nn.MSELoss()
ac_optimizer = torch.optim.Adam(ac_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

ac_model.train()

batch  = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    
    mfccs = get_mfccs(ac_f, start, end).T

    if mfccs is None:
        continue

    batch.append(
        torch.tensor(mfccs, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1
    ac_batch = torch.stack(batch).to(device)  # (B,T,F)
    batch.clear()

    # ===== Acoustic =====
    ac_out, ac_z = ac_model(ac_batch)
    ac_loss, ac_static, ac_delta = combined_loss(
        ac_out, ac_batch
    )

    ac_optimizer.zero_grad()
    ac_loss.backward()
    ac_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(ac_loss.item())
    pbar.set_description(
        f"{batch_idx}: AC {ac_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if batch_idx % 1000 == 999:
        with torch.no_grad():
            ac_stats  = latent_stats(ac_z)

        print(
            f"[AC ] latent var mean={ac_stats['latent_var_mean']:.4e}, "
            f"min={ac_stats['latent_var_min']:.4e}"
        )
        print(
            f"[AC ] input std={batch_diversity(ac_batch):.4e}, "
            f"recon std={batch_diversity(ac_out):.4e}"
        )

        torch.save(ac_model.state_dict(), "ckpts/aemri/ac_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/aemri/ac_model_training_log.csv", index=False)

torch.save(ac_model.state_dict(), "ckpts/aemri/ac_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/aemri/ac_model_training_log.csv", index=False)

999: AC 164.7690:   1%|          | 979/129384 [00:05<09:44, 219.67it/s] 

[AC ] latent var mean=1.2359e+03, min=8.2164e-04
[AC ] input std=1.4405e+01, recon std=4.0498e+00


1999: AC 126.9526:   2%|▏         | 1996/129384 [00:10<10:01, 211.63it/s]

[AC ] latent var mean=2.9891e+03, min=5.3760e-03
[AC ] input std=1.3895e+01, recon std=5.8752e+00


2999: AC 133.7991:   2%|▏         | 2992/129384 [00:15<09:24, 224.04it/s] 

...

127999: AC 93.8624:  99%|█████████▉| 127992/129384 [13:01<00:06, 221.48it/s] 

[AC ] latent var mean=1.1368e+08, min=2.1283e+02
[AC ] input std=1.2323e+01, recon std=5.8093e+00


128999: AC 92.3924: 100%|█████████▉| 128984/129384 [13:06<00:01, 226.72it/s] 

[AC ] latent var mean=7.8645e+07, min=3.9850e+03
[AC ] input std=1.4480e+01, recon std=6.2095e+00


129383: AC 80.4298: 100%|██████████| 129384/129384 [13:08<00:00, 164.13it/s] 
